In [1]:
import json
import pandas as pd
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader, random_split


with open("../data/tokenizer.json", "r") as f:
    token_to_id = json.load(f)

if "<PAD>" not in token_to_id:
    token_to_id["<PAD>"] = max(token_to_id.values()) + 1
pad_idx = token_to_id["<PAD>"]

def tokenize(word, token_to_id):
    """
    Greedy longest-match tokenization.
    
    Args:
        word (str): The word to tokenize.
        token_to_id (dict): Mapping from token string to unique integer ID.
    
    Returns:
        List[int]: List of token IDs.
    """
    tokens = []
    i = 0
    sorted_tokens = sorted(token_to_id.keys(), key=len, reverse=True)
    while i < len(word):
        match = None
        for token in sorted_tokens:
            if token == "<PAD>":
                continue
            if word[i:i+len(token)] == token:
                match = token
                break
        if match is None:
            match = word[i]
            i += 1
        else:
            i += len(match)
        tokens.append(token_to_id[match])
    return tokens

letter_to_idx = {letter: idx for idx, letter in enumerate("abcdefghijklmnopqrstuvwxyz")}
idx_to_letter = {idx: letter for letter, idx in letter_to_idx.items()}

In [2]:


class HangmanDataset(Dataset):
    def __init__(self, dataframe, tokenizer, max_seq_len=20):
        """
        Args:
            dataframe (pd.DataFrame): DataFrame with columns "masked_word", "masked_letter", "original_word".
            tokenizer (function): A function that takes a word and returns a list of token IDs.
            max_seq_len (int): Maximum sequence length for padding/truncation.
        """
        self.data = dataframe
        self.tokenizer = lambda word: tokenize(word, token_to_id)
        self.max_seq_len = max_seq_len

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        row = self.data.iloc[idx]
        masked_word = row["masked_word"].strip().lower()
        masked_letter = row["masked_letter"].strip().lower()
        
        token_ids = self.tokenizer(masked_word)
        if len(token_ids) < self.max_seq_len:
            token_ids = token_ids + [pad_idx] * (self.max_seq_len - len(token_ids))
        else:
            token_ids = token_ids[:self.max_seq_len]
        token_ids = torch.tensor(token_ids, dtype=torch.long)
        target = letter_to_idx[masked_letter]
        target = torch.tensor(target, dtype=torch.long)
        return token_ids, target

df = pd.read_csv("../data/masked_words_dataset.csv")
dataset = HangmanDataset(df, tokenize, max_seq_len=20)
train_size = int(0.8 * len(dataset))
val_size = len(dataset) - train_size
train_dataset, val_dataset = random_split(dataset, [train_size, val_size])
train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=32, shuffle=False)


In [3]:
class HangmanLSTM(nn.Module):
    def __init__(self, vocab_size, embedding_dim, hidden_dim, output_dim, mask_token):
        """
        Args:
            vocab_size (int): Size of the vocabulary.
            embedding_dim (int): Embedding vector dimension.
            hidden_dim (int): Hidden dimension for the LSTM.
            output_dim (int): Number of target classes (e.g., 26 letters).
            mask_token (str): The token used for masking (e.g., "_" ).
        """
        super(HangmanLSTM, self).__init__()
        self.embedding = nn.Embedding(vocab_size, embedding_dim, padding_idx=pad_idx)
        #self.conv1 = nn.Conv1d(embedding_dim, 128, kernel_size=3, padding=1)
        self.lstm = nn.LSTM(embedding_dim, hidden_dim, num_layers=2, bidirectional=True, batch_first=True)
        self.fc = nn.Linear(hidden_dim * 2, output_dim)
        self.mask_idx = token_to_id.get(mask_token, None)
        if self.mask_idx is None:
            raise ValueError(f"Mask token '{mask_token}' not found in tokenizer.")
    
    def forward(self, input_ids):
        embeds = self.embedding(input_ids) 
        #conved = self.conv1(embeds.transpose(1, 2))
        #conved = torch.relu(conved)
        #conved = conved.transpose(1, 2)
        #lstm_out, _ = self.lstm(conved)
        lstm_out, _ = self.lstm(embeds)       
    
        mask_positions = (input_ids == self.mask_idx)  
        mask_positions_float = mask_positions.float()
        
        sum_masked = torch.sum(lstm_out * mask_positions_float.unsqueeze(-1), dim=1)  
        count_masked = torch.sum(mask_positions_float, dim=1, keepdim=True).clamp(min=1)  
        avg_masked = sum_masked / count_masked
        
        logits = self.fc(avg_masked)
        return logits

In [4]:
vocab_size = len(token_to_id)
embedding_dim = 50
hidden_dim = 128
output_dim = 26  
mask_token = "_"
model = HangmanLSTM(vocab_size, embedding_dim, hidden_dim, output_dim, mask_token)

def train_model(model, train_loader, val_loader, num_epochs=50, learning_rate=0.001, patience=5, device="cpu"):
    criterion = nn.CrossEntropyLoss()
    optimizer = optim.Adam(model.parameters(), lr=learning_rate)
    model.to(device)
    best_val_loss = float("inf")
    epochs_no_improve = 0
    for epoch in range(num_epochs):
        model.train()
        running_loss = 0.0
        for inputs, targets in train_loader:
            inputs,  targets = inputs.to(device), targets.to(device)
            optimizer.zero_grad()
            outputs = model(inputs)
            loss = criterion(outputs, targets)
            loss.backward()
            optimizer.step()
            running_loss += loss.item()
        avg_train_loss = running_loss / len(train_loader)
        model.eval()
        val_loss = 0.0
        with torch.no_grad():
            for inputs,  targets in val_loader:
                inputs, targets = inputs.to(device),  targets.to(device)
                outputs = model(inputs)
                loss = criterion(outputs, targets)
                val_loss += loss.item()
        avg_val_loss = val_loss / len(val_loader)
        print(f"Epoch {epoch+1} Train Loss: {avg_train_loss:.4f} Val Loss: {avg_val_loss:.4f}")
        if avg_val_loss < best_val_loss:
            best_val_loss = avg_val_loss
            epochs_no_improve = 0
            torch.save(model.state_dict(), "../models/bpe_lstm.pth")
        else:
            epochs_no_improve += 1
        if epochs_no_improve >= patience:
            print("Early stopping triggered")
            break

train_model(model, train_loader, val_loader, num_epochs=5, learning_rate=0.001)

/Users/apple/Library/Python/3.9/lib/python/site-packages/urllib3/__init__.py:34: NotOpenSSLWarning: urllib3 v2.0 only supports OpenSSL 1.1.1+, currently the 'ssl' module is compiled with 'LibreSSL 2.8.3'. See: https://github.com/urllib3/urllib3/issues/3020
  warnings.warn(


Epoch 1 Train Loss: 1.1700 Val Loss: 0.9740
Epoch 2 Train Loss: 0.8973 Val Loss: 0.9017
Epoch 3 Train Loss: 0.8208 Val Loss: 0.8792
Epoch 4 Train Loss: 0.7778 Val Loss: 0.8643
Epoch 5 Train Loss: 0.7486 Val Loss: 0.8574
